In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# 🔥 1. 데이터 (작고 noisy하게)
np.random.seed(42)

X = np.linspace(0, 1, 50) # 0부터 1까지를 50개로 균등하게 나눔 #[0.0, 0.0204, 0.0408, ..., 1.0]
y = 3 * X + np.random.normal(0, 0.1, 50)  # noise 추가 = y = 직선 + 약간의 흔들림

X = X.reshape(-1, 1)
print(X.shape)
print(X)


# 🔥 2. 모델 (과하게 크게 만들어서 overfitting 유도)
model = Sequential([
    Dense(128, activation='relu', input_shape=(1,)),
    Dense(128, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse'
)

# val_loss를 계속 관찰한다.
# val_loss가 개선되지 않으면
# 5 Epoch 동안 기다린다.
# 개선되지 않으면 학습을 중단
# EarlyStopping 쓰는 이유는 = 과적합 해결
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

#체크 포인트 => 최고 성능 모델을 저장하는 역할
checkpoint = ModelCheckpoint(
    "best_model.keras",        # 저장 파일
    monitor='val_loss',        # 기준
    save_best_only=True,       # 최고 성능만 저장
    mode='min',                # loss는 최소값 기준
    verbose=1
)


# 학습
history = model.fit(
    X, y,
    epochs=500,
    validation_split=0.3,
    callbacks=[early_stop],
    verbose=1
)

| Epoch | val_loss | 개선 여부     |
| ----- | -------- | --------- |
| 1     | 0.65     | ✔         |
| 2     | 0.52     | ✔         |
| 3     | 0.45     | ✔         |
| 4     | 0.40     | ✔ (최고 성능) |
| 5     | 0.42     | ❌         |
| 6     | 0.44     | ❌         |
| 7     | 0.43     | ❌         |
| 8     | 0.41     | ❌         |
| 9     | 0.46     | ❌         |
| 10    | 중단       | -         |


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')

plt.title('Loss vs Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()